# Lab 1: Log Analytics with Resilient Distributed Datasets

## Team Members:
- 1.) Name: Sorawit Chaithong ID: 67070503442
- 2.) Name: Kittiphat Noikate ID: 67070503459
- 3.) Name: Piti Srisongkram  ID: 67070503467

---

## 1.) Ingestion and Data Analysis

We establish the runtime connection to Apache Spark via `SparkContext`, load the cluster execution log (`spark.log`) and analyze the structural layout of the raw semi-structured text.

### 1.1) Safe SparkContext Initialization
We configure the `SparkContext` singleton safely using `SparkContext.getOrCreate()`. Python worker executable paths are resolved dynamically. Logging level is set to `WARN` to suppress driver verbosity.

In [4]:
import os
import sys
import re

# Resolve Python worker path
def get_safe_python_path(path):
    if os.name == 'nt':
        try:
            import ctypes
            buf = ctypes.create_unicode_buffer(500)
            res = ctypes.windll.kernel32.GetShortPathNameW(path, buf, 500)
            if res > 0:
                return buf.value
        except Exception:
            pass
    return path

safe_python = get_safe_python_path(sys.executable)
os.environ["PYSPARK_PYTHON"] = safe_python
os.environ["PYSPARK_DRIVER_PYTHON"] = safe_python

from pyspark import SparkContext, SparkConf

# Configure SparkContext parameters
conf = SparkConf()
conf.setAppName("Lab7_Part1_LogAnalytics")
conf.setMaster("local[*]")
conf.set("spark.pyspark.python", safe_python)
conf.set("spark.pyspark.driver.python", safe_python)

# Cleanly terminate any dangling background context
if 'sc' in locals() or 'sc' in globals():
    try:
        sc.stop()
    except:
        pass

sc = SparkContext.getOrCreate(conf=conf)
sc.setLogLevel("WARN")

# Output cluster runtime parameters
print(f"Spark Version       : {sc.version}")
print(f"Master URL          : {sc.master}")
print(f"Application Name    : {sc.appName}")
print(f"Default Parallelism : {sc.defaultParallelism}")

Spark Version       : 4.2.0
Master URL          : local[*]
Application Name    : Lab7_Part1_LogAnalytics
Default Parallelism : 20


### 1.2) Raw Log Ingestion via `sc.textFile()`
We load `spark.log` into a distributed RDD. The text is partitioned across executors and loaded lazily.

In [5]:
# Ingest raw cluster log file
LOG_FILE_PATH = "spark.log"
raw_log_rdd = sc.textFile(LOG_FILE_PATH)

# Total lines in the raw dataset
total_raw_lines = raw_log_rdd.count()
print(f"Total lines ingested from '{LOG_FILE_PATH}': {total_raw_lines}")

Total lines ingested from 'spark.log': 2000


### 1.3) Sample Inspection with `.take(n)`
We sample the first 5 records using `.take(5)` to avoid driver memory saturation (`collect()`).

In [6]:
# Inspect sample lines using .collect()[:5]
sample_lines = raw_log_rdd.collect()[:5]
for idx, line in enumerate(sample_lines):
    print(f"[{idx}] {line}")

[0] 17/06/09 20:10:40 INFO executor.CoarseGrainedExecutorBackend: Registered signal handlers for [TERM, HUP, INT]
[1] 17/06/09 20:10:40 INFO spark.SecurityManager: Changing view acls to: yarn,curi
[2] 17/06/09 20:10:40 INFO spark.SecurityManager: Changing modify acls to: yarn,curi
[3] 17/06/09 20:10:40 INFO spark.SecurityManager: SecurityManager: authentication disabled; ui acls disabled; users with view permissions: Set(yarn, curi); users with modify permissions: Set(yarn, curi)
[4] 17/06/09 20:10:41 INFO spark.SecurityManager: Changing view acls to: yarn,curi


### 1.4) Establishing Schema and Tokenization Layout
By inspecting the log lines, we deduce the standard Apache Spark cluster log structure:
$$\text{Format: } \underbrace{\text{YY/MM/DD}}_{\text{Index 0: Date}} \quad \underbrace{\text{HH:MM:SS}}_{\text{Index 1: Time}} \quad \underbrace{\text{LEVEL}}_{\text{Index 2: Log Level}} \quad \underbrace{\text{Source:}}_{\text{Index 3: Component Source}} \quad \underbrace{\text{Payload}}_{\text{Index 4+: Message}}$$

- **Index 0:** Date (e.g., `17/06/09`)
- **Index 1:** Time (e.g., `20:10:40`)
- **Index 2:** Log severity level (`INFO`, `WARN`, `ERROR`, `DEBUG`)
- **Index 3:** Originating subsystem/class (`executor.Executor:`, `python.PythonRunner:`)
- **Index 4+:** Message payload containing task metrics, block identifiers, or error messages.

## 2.) Design RDD Transformation DAG

To construct an optimal, resilient processing DAG:
1. **Defensive Parsing Closure:** `parse_log_line(line)` inspects each line, checks minimum token bounds, validates log severity, strips trailing formatting colons (`:`) from source names, and handles errors gracefully by returning `None`.
2. **Single-Pass Projection:** We map raw lines into clean structured tuples `(clean_source, message, level, date, time)` in a single pass.
3. **Partition Caching:** The sanitized structured RDD is cached (`.cache()`) in memory because it serves as the foundational parent for Deliverables.

In [7]:
VALID_LOG_LEVELS = {"INFO", "WARN", "ERROR", "DEBUG", "TRACE", "FATAL"}

def parse_log_line(line):
    """
    Defensive parser for Spark cluster log lines.
    Returns clean_source, message, level, date, time or None if malformed.
    """
    if line is None:
        return None
    clean_line = line.strip()
    if not clean_line or clean_line.startswith("#"):
        return None
    parts = clean_line.split(maxsplit=4)
    if len(parts) < 4:
        return None
    date_str, time_str, level_str, raw_source = parts[0], parts[1], parts[2], parts[3]
    if level_str not in VALID_LOG_LEVELS:
        return None
    clean_source = raw_source.rstrip(":").strip()
    message = parts[4].strip() if len(parts) > 4 else ""
    return (clean_source, message, level_str, date_str, time_str)

# Build the foundational parsed RDD
parsed_log_rdd = (
    raw_log_rdd
    .map(parse_log_line)
    .filter(lambda record: record is not None)
    .cache()
)

valid_record_count = parsed_log_rdd.count()
print(f"Total raw lines    : {total_raw_lines}")
print(f"Valid parsed lines : {valid_record_count}")
print(f"Filtered / Ignored : {total_raw_lines - valid_record_count}")

Total raw lines    : 2000
Valid parsed lines : 2000
Filtered / Ignored : 0


In [8]:
# Inspect sample standardized parsed tuples
# Schema: (clean_source, message, level, date, time)
for rec in parsed_log_rdd.collect()[:3]:
    print(f"Source : '{rec[0]}'")
    print(f"Message: '{rec[1]}'")
    print(f"Meta   : Level={rec[2]}, Date={rec[3]}, Time={rec[4]}\n")

Source : 'executor.CoarseGrainedExecutorBackend'
Message: 'Registered signal handlers for [TERM, HUP, INT]'
Meta   : Level=INFO, Date=17/06/09, Time=20:10:40

Source : 'spark.SecurityManager'
Message: 'Changing view acls to: yarn,curi'
Meta   : Level=INFO, Date=17/06/09, Time=20:10:40

Source : 'spark.SecurityManager'
Message: 'Changing modify acls to: yarn,curi'
Meta   : Level=INFO, Date=17/06/09, Time=20:10:40



## 3.) Implementation

We implement the three required deliverables using pure PySpark Core RDD operations.

### Deliverable 1.1: Source Frequency Report

Compute the total frequency of occurrences for every distinct source component in the log.

**Spark Optimization Strategy:**
- Project records into key-value pairs `(source_name, 1)`.
- Aggregate with `reduceByKey(lambda a, b: a + b)` instead of `groupByKey()`.
  > **Architectural Note:** `reduceByKey` executes **Map-Side Combining** inside each worker partition before shuffling data across the network. This minimizes network I/O, serialization overhead, and garbage collection pressure.
- Sort in descending order of frequency using `sortBy(lambda x: x[1], ascending=False)`.

In [9]:
# Source Frequency Transformation Pipeline
source_frequency_rdd = (
    parsed_log_rdd
    .map(lambda x: (x[0], 1))
    .reduceByKey(lambda a, b: a + b)
    .sortBy(lambda x: x[1], ascending=False)
)

# Fetch sorted frequency list
source_frequencies = source_frequency_rdd.collect()
total_events = sum(count for _, count in source_frequencies)

print("=" * 72)
print(f"{'RANK':<5} | {'SOURCE COMPONENT':<42} | {'COUNT':<8} | {'PERCENT':<8}")
print("=" * 72)
for rank, (src, cnt) in enumerate(source_frequencies, start=1):
    pct = (cnt / total_events) * 100
    print(f"{rank:<5} | {src:<42} | {cnt:<8} | {pct:>6.2f}%")
print("=" * 72)
print(f"{'TOTAL':<48} | {total_events:<8} | 100.00%")
print("=" * 72)

RANK  | SOURCE COMPONENT                           | COUNT    | PERCENT 
1     | executor.Executor                          | 606      |  30.30%
2     | python.PythonRunner                        | 375      |  18.75%
3     | executor.CoarseGrainedExecutorBackend      | 308      |  15.40%
4     | storage.BlockManager                       | 257      |  12.85%
5     | storage.MemoryStore                        | 150      |   7.50%
6     | spark.CacheManager                         | 75       |   3.75%
7     | broadcast.TorrentBroadcast                 | 74       |   3.70%
8     | output.FileOutputCommitter                 | 60       |   3.00%
9     | rdd.HadoopRDD                              | 45       |   2.25%
10    | mapred.SparkHadoopMapRedUtil               | 30       |   1.50%
11    | spark.SecurityManager                      | 6        |   0.30%
12    | Configuration.deprecation                  | 5        |   0.25%
13    | storage.BlockManagerMaster                 | 2        |

### Benchmark RDD Operations Comparison
To align with the techniques demonstrated in lecture, we also implement:
1. `source_info_rdd` pair RDD and `.keys()`, `.values()`.
2. Calculating total word counts per source (`source_nword_rdd`).
3. Calculating line counts per source (`source_nline_rdd`).
4. Joining line counts and word counts via `.join()`.
5. Average message length per source (`source_avglen_rdd`), verifying the **13.0** word benchmark for `python.PythonRunner`.

In [10]:
# Source info and message word count
source_info_rdd = parsed_log_rdd.map(lambda x: (x[0], x[1]))
source_len_rdd = source_info_rdd.mapValues(lambda msg: len(msg.split()))

# Total words across all messages in cluster
total_words = source_len_rdd.values().reduce(lambda a, b: a + b)
print(f"Total words across all log messages: {total_words}")

# Total words per source using reduceByKey
source_nword_rdd = source_len_rdd.reduceByKey(lambda a, b: a + b)

# Total lines per source using mapValues and reduceByKey
source_nline_rdd = source_len_rdd.mapValues(lambda _: 1).reduceByKey(lambda a, b: a + b)

# Join lines and word counts: (source, (line_count, word_count))
joined_metrics_rdd = source_nline_rdd.join(source_nword_rdd)

# Compute average words per line sorted descending
source_avglen_rdd = joined_metrics_rdd.mapValues(lambda stats: stats[1] / stats[0])
sorted_avglen = source_avglen_rdd.sortBy(lambda x: x[1], ascending=False).take(10)

print("\n[Top 10 Sources by Average Message Word Count]")
for src, avg_len in sorted_avglen:
    print(f"{src:<42} : {avg_len:.2f} words/line")

Total words across all log messages: 17511

[Top 10 Sources by Average Message Word Count]
storage.MemoryStore                        : 13.95 words/line
python.PythonRunner                        : 13.00 words/line
executor.Executor                          : 10.97 words/line
spark.SecurityManager                      : 9.33 words/line
output.FileOutputCommitter                 : 7.00 words/line
util.Utils                                 : 7.00 words/line
broadcast.TorrentBroadcast                 : 6.00 words/line
spark.CacheManager                         : 6.00 words/line
Configuration.deprecation                  : 6.00 words/line
storage.DiskBlockManager                   : 5.00 words/line


### Deliverable 1.2: Statistical Analysis and Outlier Detection for `python.PythonRunner`

Isolate logs produced by `python.PythonRunner` and compute comprehensive descriptive statistics and statistical outliers.

**Dual-Metric Analysis:**
1. **Structural Metric:** Message word count `len(message.split())` reproducing the slide benchmark of exactly 13.0 words.
2. **Primary Operational Metric:** Total task execution duration in milliseconds extracted via Regex (`total = (\d+)`) from the PySpark worker heartbeat payload.

**Outlier Detection Methodology:**
- Compute Quartiles: $Q1$ (25th percentile) and $Q3$ (75th percentile) using distributed RDD sorting (`sortBy`) and indexing (`zipWithIndex`).
- Compute Interquartile Range:
  $$IQR = Q3 - Q1$$
- Determine Outer Bounds:
  $$\text{Lower Bound} = Q1 - 1.5 \times IQR$$
  $$\text{Upper Bound} = Q3 + 1.5 \times IQR$$
- Isolate and enumerate outliers: Values outside $[\text{Lower Bound}, \text{Upper Bound}]$. 

In [11]:
# Filter RDD for python.PythonRunner
py_runner_rdd = parsed_log_rdd.filter(lambda x: x[0] == "python.PythonRunner").cache()
py_runner_count = py_runner_rdd.count()
print(f"Total 'python.PythonRunner' entries: {py_runner_count}")

Total 'python.PythonRunner' entries: 375


In [12]:
# Metric A: Structural Payload Word Count Metric
py_word_counts_rdd = py_runner_rdd.map(lambda x: len(x[1].split())).cache()

word_n = py_word_counts_rdd.count()
word_min = py_word_counts_rdd.min()
word_max = py_word_counts_rdd.max()
word_mean = py_word_counts_rdd.mean()
word_stdev = py_word_counts_rdd.stdev()

print("[Metric A: python.PythonRunner Payload Word Count Statistics]")
print(f"Count (N)         : {word_n}")
print(f"Minimum Words     : {word_min}")
print(f"Maximum Words     : {word_max}")
print(f"Mean Words/Line   : {word_mean:.4f}")
print(f"Standard Deviation: {word_stdev:.4f}")

[Metric A: python.PythonRunner Payload Word Count Statistics]
Count (N)         : 375
Minimum Words     : 13
Maximum Words     : 13
Mean Words/Line   : 13.0000
Standard Deviation: 0.0000


In [13]:
# Metric B: Primary Operational Metric
def extract_total_duration(msg):
    match = re.search(r"total\s*=\s*(-?\d+)", msg)
    if match:
        return int(match.group(1))
    return None

py_duration_rdd = (
    py_runner_rdd
    .map(lambda x: extract_total_duration(x[1]))
    .filter(lambda x: x is not None)
    .cache()
)

# Compute descriptive statistics using RDD actions
N = py_duration_rdd.count()
dur_min = py_duration_rdd.min()
dur_max = py_duration_rdd.max()
dur_mean = py_duration_rdd.mean()
dur_stdev = py_duration_rdd.stdev()
dur_variance = py_duration_rdd.variance()

print("[Metric B: python.PythonRunner Total Execution Time (ms)]")
print(f"Sample Size (N)       : {N}")
print(f"Minimum Duration      : {dur_min} ms")
print(f"Maximum Duration      : {dur_max} ms")
print(f"Arithmetic Mean       : {dur_mean:.2f} ms")
print(f"Variance              : {dur_variance:.2f} ms^2")
print(f"Standard Deviation    : {dur_stdev:.2f} ms")

[Metric B: python.PythonRunner Total Execution Time (ms)]
Sample Size (N)       : 375
Minimum Duration      : 37 ms
Maximum Duration      : 1114 ms
Arithmetic Mean       : 55.66 ms
Variance              : 14295.63 ms^2
Standard Deviation    : 119.56 ms


#### Distributed Quartile and Outlier Computation using Pure RDD Operations
We compute $Q1$ and $Q3$ distributively using `sortBy()`, `zipWithIndex()`, and key-value indexing without moving data to pandas or DataFrames.

In [14]:
# Sort the numerical RDD ascending
sorted_durations = py_duration_rdd.sortBy(lambda val: val, ascending=True)

# Assign 0-based indices: (index, duration_value)
indexed_durations = (
    sorted_durations
    .zipWithIndex()
    .map(lambda item: (item[1], item[0])) # (index, value)
    .cache()
)

# Determine index positions for Q1 (25%) and Q3 (75%)
q1_idx = int(round(0.25 * (N - 1)))
q3_idx = int(round(0.75 * (N - 1)))

# Retrieve Q1 and Q3 via RDD lookup
Q1 = indexed_durations.lookup(q1_idx)[0]
Q3 = indexed_durations.lookup(q3_idx)[0]

# Compute IQR and Tukey's Fence bounds
IQR = Q3 - Q1
lower_bound = Q1 - 1.5 * IQR
upper_bound = Q3 + 1.5 * IQR

# Filter outliers using RDD transformations
outliers_rdd = py_duration_rdd.filter(lambda val: val < lower_bound or val > upper_bound).cache()
outlier_count = outliers_rdd.count()
outlier_percentage = (outlier_count / N) * 100

print("=" * 60)
print("DISTRIBUTED OUTLIER DETECTION REPORT")
print("=" * 60)
print(f"First Quartile (Q1, 25th percentile) : {Q1} ms")
print(f"Third Quartile (Q3, 75th percentile) : {Q3} ms")
print(f"Interquartile Range (IQR)           : {IQR} ms")
print(f"Lower Bound (Q1 - 1.5 * IQR)        : {lower_bound:.2f} ms")
print(f"Upper Bound (Q3 + 1.5 * IQR)        : {upper_bound:.2f} ms")
print("-" * 60)
print(f"Total Outliers Detected             : {outlier_count} ({outlier_percentage:.2f}%)")
print("=" * 60)

DISTRIBUTED OUTLIER DETECTION REPORT
First Quartile (Q1, 25th percentile) : 39 ms
Third Quartile (Q3, 75th percentile) : 42 ms
Interquartile Range (IQR)           : 3 ms
Lower Bound (Q1 - 1.5 * IQR)        : 34.50 ms
Upper Bound (Q3 + 1.5 * IQR)        : 46.50 ms
------------------------------------------------------------
Total Outliers Detected             : 46 (12.27%)


In [15]:
# Isolate and inspect top extreme outliers
extreme_outliers = (
    outliers_rdd
    .map(lambda x: (x, 1))
    .reduceByKey(lambda a, b: a + b)
    .sortBy(lambda x: x[0], ascending=False)
    .collect()
)

print(f"{'DURATION (ms)':<15} | {'FREQUENCY':<10} | {'OUTLIER CLASSIFICATION'}")
print("-" * 55)
for dur, count in extreme_outliers[:10]:
    category = "Extreme High (Process Boot / Cold Start)" if dur > 500 else "Moderate High (GC / CPU Spikes)"
    print(f"{dur:<15} | {count:<10} | {category}")

DURATION (ms)   | FREQUENCY  | OUTLIER CLASSIFICATION
-------------------------------------------------------
1114            | 1          | Extreme High (Process Boot / Cold Start)
1078            | 1          | Extreme High (Process Boot / Cold Start)
1077            | 1          | Extreme High (Process Boot / Cold Start)
1074            | 1          | Extreme High (Process Boot / Cold Start)
1072            | 1          | Extreme High (Process Boot / Cold Start)
108             | 1          | Moderate High (GC / CPU Spikes)
65              | 1          | Moderate High (GC / CPU Spikes)
60              | 1          | Moderate High (GC / CPU Spikes)
59              | 1          | Moderate High (GC / CPU Spikes)
58              | 2          | Moderate High (GC / CPU Spikes)


### Deliverable 1.3: Fault Detection (Lost Tasks and Stages Analysis)

Identify, extract and categorize task and stage failure incidents originating from `executor.Executor`.

**Root-Cause Keywords:**
Spark executors log task eviction, communication loss, and shuffle errors using standard patterns:
- `"lost"`
- `"Lost task"`
- `"Stage lost"`
- `"FetchFailed"`

**Parsing Requirements:**
We extract:
- Incident Type (`TASK_LOST` or `STAGE_LOST`)
- Task Identifier (`Task ID` / `TID`)
- Stage Identifier & Attempt Number
- Root-Cause Classification (e.g., `FetchFailedException`, `OutOfMemoryError`, `YARN Container Exceeded Memory`, `Network Timeout`).

In [16]:
# Filter parsed RDD for executor.Executor
executor_rdd = parsed_log_rdd.filter(lambda x: x[0] == "executor.Executor").cache()
executor_total = executor_rdd.count()

# Filter for failure/loss indicators
loss_keywords = ["lost", "lost task", "stage lost", "fetchfailed"]
fault_rdd = (
    executor_rdd
    .filter(lambda x: any(kw in x[1].lower() for kw in loss_keywords))
    .cache()
)
fault_count = fault_rdd.count()

print(f"Total executor.Executor entries : {executor_total}")
print(f"Fault / Loss entries detected   : {fault_count}")

Total executor.Executor entries : 606
Fault / Loss entries detected   : 0


In [17]:
# Parser function for lost task and lost stage log records
def parse_executor_fault(record):
    """
    Extracts structured failure metadata from executor fault messages.
    record: (source, message, level, date, time)
    """
    src, msg, lvl, dt, tm = record
    timestamp = f"{dt} {tm}"
    
    # Pattern 1: Lost Task
    # Example: Lost task 142.0 in stage 3.0 (TID 512, 10.35.23.230, executor 2): FetchFailed()
    task_match = re.search(r"Lost task\s+([\d\.]+)\s+in stage\s+([\d\.]+)\s*(?:\(([^)]+)\))?:\s*(.*)", msg, re.IGNORECASE)
    if task_match:
        task_id = task_match.group(1)
        stage_id = task_match.group(2)
        exec_meta = task_match.group(3) if task_match.group(3) else "N/A"
        reason = task_match.group(4).strip()
        return ("TASK_LOST", task_id, stage_id, exec_meta, reason, timestamp)
    
    # Pattern 2: Stage Lost
    # Example: Stage lost: 3.0 (TID 512) due to FetchFailed()
    stage_match = re.search(r"Stage lost:?\s*([\d\.]+)\s*(?:\([^)]*\))?\s*due to\s*(.*)", msg, re.IGNORECASE)
    if stage_match:
        stage_id = stage_match.group(1)
        reason = stage_match.group(2).strip()
        return ("STAGE_LOST", "N/A", stage_id, "Cluster-wide", reason, timestamp)
    
    return ("GENERIC_FAULT", "N/A", "N/A", "N/A", msg, timestamp)

parsed_faults_rdd = fault_rdd.map(parse_executor_fault).cache()
fault_list = parsed_faults_rdd.collect()

print("=" * 110)
print(f"{'INCIDENT':<12} | {'TASK ID':<8} | {'STAGE':<8} | {'TIMESTAMP':<18} | {'FAILURE REASON'}")
print("=" * 110)
if len(fault_list) == 0:
    print("STATUS: All 606 executor tasks completed successfully (0 task/stage loss detected).")
    print("Health Assessment: Healthy cluster run with zero lost stages.")
else:
    for f in fault_list:
        print(f"{f[0]:<12} | {f[1]:<8} | {f[2]:<8} | {f[5]:<18} | {f[4][:55]}")
print("=" * 110)

INCIDENT     | TASK ID  | STAGE    | TIMESTAMP          | FAILURE REASON
STATUS: All 606 executor tasks completed successfully (0 task/stage loss detected).
Health Assessment: Healthy cluster run with zero lost stages.


### 3.4) Fault Detection Engine Verification with Incident Simulation
To prove that our fault detection parser accurately identifies and extracts lost tasks and stages in cluster failure scenarios, we benchmark the extraction engine against a synthesized cluster incident log RDD.

In [18]:
# Benchmark incident log RDD demonstrating all canonical Spark loss scenarios
incident_raw_data = [
    "17/06/09 20:14:02 WARN executor.Executor: Lost task 142.0 in stage 3.0 (TID 512, 10.35.23.230, executor 2): FetchFailed(BlockManagerId(1, 10.35.23.231, 7337), shuffleId=1, mapId=24, reduceId=12, message=org.apache.spark.shuffle.FetchFailedException: Failed to connect to /10.35.23.231:7337)",
    "17/06/09 20:14:05 ERROR executor.Executor: Lost task 89.1 in stage 2.0 (TID 305, 10.35.23.229, executor 1): java.lang.OutOfMemoryError: Java heap space",
    "17/06/09 20:15:12 WARN executor.Executor: Stage lost: 3.0 (TID 512) due to FetchFailed(BlockManagerId(1, 10.35.23.231, 7337))",
    "17/06/09 20:16:44 ERROR executor.Executor: Lost task 210.0 in stage 4.0 (TID 620, 10.35.23.232, executor 3): ExecutorLostFailure (executor 3 exited caused by one of the running tasks) Reason: Container killed by YARN for exceeding memory limits. 2.1 GB of 2.0 GB physical memory used.",
    "17/06/09 20:17:01 WARN executor.Executor: Lost task 45.0 in stage 1.0 (TID 188, 10.35.23.230, executor 2): FetchFailed(BlockManagerId(3, 10.35.23.232, 7337), shuffleId=0, mapId=11, reduceId=5, message=Connection reset by peer)",
    "17/06/09 20:17:05 WARN executor.Executor: Stage lost: 1.0 (TID 188) due to FetchFailed(BlockManagerId(3, 10.35.23.232, 7337))",
    "17/06/09 20:18:10 ERROR executor.Executor: Lost task 315.0 in stage 5.0 (TID 840, 10.35.23.231, executor 4): java.io.IOException: Connection to 10.35.23.230:42117 timed out after 120 seconds"
]

incident_rdd = (
    sc.parallelize(incident_raw_data)
    .map(parse_log_line)
    .filter(lambda x: x is not None)
    .map(parse_executor_fault)
)

benchmark_incidents = incident_rdd.collect()

print("=" * 115)
print(f"{'TYPE':<12} | {'TASK ID':<8} | {'STAGE':<7} | {'HOST / METADATA':<32} | {'REASON / ROOT CAUSE'}")
print("=" * 115)
for inc in benchmark_incidents:
    print(f"{inc[0]:<12} | {inc[1]:<8} | {inc[2]:<7} | {inc[3][:32]:<32} | {inc[4][:48]}")
print("=" * 115)

TYPE         | TASK ID  | STAGE   | HOST / METADATA                  | REASON / ROOT CAUSE
TASK_LOST    | 142.0    | 3.0     | TID 512, 10.35.23.230, executor  | FetchFailed(BlockManagerId(1, 10.35.23.231, 7337
TASK_LOST    | 89.1     | 2.0     | TID 305, 10.35.23.229, executor  | java.lang.OutOfMemoryError: Java heap space
STAGE_LOST   | N/A      | 3.0     | Cluster-wide                     | FetchFailed(BlockManagerId(1, 10.35.23.231, 7337
TASK_LOST    | 210.0    | 4.0     | TID 620, 10.35.23.232, executor  | ExecutorLostFailure (executor 3 exited caused by
TASK_LOST    | 45.0     | 1.0     | TID 188, 10.35.23.230, executor  | FetchFailed(BlockManagerId(3, 10.35.23.232, 7337
STAGE_LOST   | N/A      | 1.0     | Cluster-wide                     | FetchFailed(BlockManagerId(3, 10.35.23.232, 7337
TASK_LOST    | 315.0    | 5.0     | TID 840, 10.35.23.231, executor  | java.io.IOException: Connection to 10.35.23.230:


## 4.) Testing and Verification

We validate edge case resilience, defensive parsing integrity and statistical stability across adverse data conditions:
1. **Malformed and Poisoned Records:** Handling empty lines, comment lines, irregular whitespaces, corrupted delimiters and truncated records.
2. **Zero-Division Handling:** Defending against empty collections or single-element RDDs when computing descriptive statistics.
3. **Data Integrity and Consistency Assertions:** Verifying that no data is silently dropped or corrupted.

In [19]:
# Test 1: Resilience against corrupted / poisoned log records
poisoned_sample_data = [
    "",
    "   ",
    "# Checkpoint comment marker",
    "CORRUPTED_ROW_WITHOUT_WHITESPACE",
    "17/06/09 20:25:00", # Truncated
    "17/06/09 20:25:00 INVALID_LEVEL Component: Message", # Unrecognized level
    "17/06/09 20:25:00 INFO spark.SecurityManager: Valid message"
]

poisoned_rdd = sc.parallelize(poisoned_sample_data)
qa_parsed = poisoned_rdd.map(parse_log_line).cache()

passed_count = qa_parsed.filter(lambda x: x is not None).count()
rejected_count = qa_parsed.filter(lambda x: x is None).count()

print("[Test 1: Defensive Parsing Audit]")
print(f"Poisoned Records Ingested : {len(poisoned_sample_data)}")
print(f"Safely Rejected Records   : {rejected_count}")
print(f"Properly Accepted Records : {passed_count}")
assert passed_count == 1, f"Expected 1 valid record, got {passed_count}"
assert rejected_count == 6, f"Expected 6 rejected records, got {rejected_count}"
print(">> Test 1 PASSED: Zero unhandled worker exceptions on malformed rows.")

[Test 1: Defensive Parsing Audit]
Poisoned Records Ingested : 7
Safely Rejected Records   : 6
Properly Accepted Records : 1
>> Test 1 PASSED: Zero unhandled worker exceptions on malformed rows.


In [20]:
# Test 2: Safe Statistical Computation and Zero-Division Defense
def safe_rdd_stats(numeric_rdd):
    """
    Computes summary statistics safely with zero-division guards.
    """
    n = numeric_rdd.count()
    if n == 0:
        return {"count": 0, "min": None, "max": None, "mean": 0.0, "stdev": 0.0}
    
    val_min = numeric_rdd.min()
    val_max = numeric_rdd.max()
    val_mean = numeric_rdd.mean()
    val_stdev = numeric_rdd.stdev() if n > 1 else 0.0
    return {"count": n, "min": val_min, "max": val_max, "mean": val_mean, "stdev": val_stdev}

# Test with empty RDD
empty_rdd = sc.emptyRDD()
empty_stats = safe_rdd_stats(empty_rdd)
print("[Test 2: Empty RDD Safe Statistics]")
print(f"Empty RDD Result: {empty_stats}")
assert empty_stats["count"] == 0
assert empty_stats["mean"] == 0.0

# Test with single-element RDD (n=1)
single_rdd = sc.parallelize([42])
single_stats = safe_rdd_stats(single_rdd)
print(f"Single Item RDD Result: {single_stats}")
assert single_stats["count"] == 1
assert single_stats["stdev"] == 0.0
print(">> Test 2 PASSED: Division-by-zero guarded.")

[Test 2: Empty RDD Safe Statistics]
Empty RDD Result: {'count': 0, 'min': None, 'max': None, 'mean': 0.0, 'stdev': 0.0}
Single Item RDD Result: {'count': 1, 'min': 42, 'max': 42, 'mean': 42.0, 'stdev': 0.0}
>> Test 2 PASSED: Division-by-zero guarded.


In [21]:
# Test 3: Precision and Consistency Assertions
# Verify sum of source frequencies equals total valid records
sum_of_frequencies = source_frequency_rdd.map(lambda x: x[1]).reduce(lambda a, b: a + b)
print("[Test 3: Totals Reconciliation]")
print(f"Valid Records in RDD : {valid_record_count}")
print(f"Sum of Aggregations  : {sum_of_frequencies}")
assert sum_of_frequencies == valid_record_count, "Data loss detected in frequency aggregation."
print(">> Test 3 PASSED: all precision with zero record loss.")

[Test 3: Totals Reconciliation]
Valid Records in RDD : 2000
Sum of Aggregations  : 2000
>> Test 3 PASSED: all precision with zero record loss.


## 5.) Documentation

### 5.1) Distributed Architecture and Transformation Strategy Analysis

| RDD Operation | Type | Shuffle Behavior | Memory & Network Impact | Recommended Usage |
| :--- | :--- | :--- | :--- | :--- |
| `reduceByKey` | Transformation | **Map-side Combine with Shuffle** | **Low:** Elements with identical keys are pre-aggregated in executor buffers before network transfer. | **Standard for Aggregations** (sums, counts, min/max). |
| `groupByKey` | Transformation | **Full Shuffle (All-to-All)** | **High:** Every individual element is transferred across the network, leading to high GC pressure and `OutOfMemoryError`. | **Avoid for aggregations**; use only when maintaining unique collections per key. |
| `mapValues` | Transformation | **Zero Shuffle (Narrow)** | **Zero network I/O:** Transformations execute locally within existing partition boundaries. | **Optimal for value manipulations** on Pair RDDs. |
| `sortBy` | Transformation | **Range Partition Shuffle** | **Moderate:** Samples data to estimate partition bounds, then performs distributed sorting. | Use for ordered reports after filtering or aggregation. |

---

### 5.2) Interpretation of Spark Cluster Faults
In Apache Spark distributed architectures, task and stage loss events originate from three primary failure mechanisms:

1. **Shuffle Fetch Failures (`FetchFailedException`):**
   - **Mechanism:** During a shuffle phase between Map and Reduce stages, downstream reducers attempt to fetch shuffle output blocks over Netty/TCP from upstream executors.
   - **Root Cause:** If an upstream worker node crashes, experiences high network packet drop or if the Netty server port times out due to GC pauses, the fetch fails.
   - **Spark Resiliency Reaction:** The `DAGScheduler` marks the stage as failed (`Stage lost`) and resubmits the upstream stage to recompute the missing partitions.

2. **JVM Heap Exhaustion (`OutOfMemoryError: Java heap space`):**
   - **Mechanism:** Spark executors allocate memory between *Execution* shuffle, join, sort buffers and *Storage* RDD caching, broadcast variables.
   - **Root Cause:** Large partitions (data skew) or high memory payloads using `groupByKey().collect()` exceed the executor JVM heap size configured by `--executor-memory`.

3. **YARN Container Eviction (`Container killed by YARN for exceeding memory limits`):**
   - **Mechanism:** In YARN managed clusters, NodeManagers monitor the total physical memory consumption of container cgroups.
   - **Root Cause:** PySpark uses off-heap memory for Python worker daemons (`python.PythonRunner`) and JVM overhead (`spark.yarn.executor.memoryOverhead`). When JVM heap with off-heap Python processes exceed the YARN allocation, YARN terminates the container immediately with `SIGKILL`.

### 5.3 Teardown of SparkContext
We release cluster resources and stop the driver daemon cleanly.

In [22]:
# Clean shutdown of SparkContext
sc.stop()
print("SparkContext shut down. All cluster resources released.")

SparkContext shut down. All cluster resources released.
